# 🚀 帧锁 (FrameSeek) - Google Colab GPU 驱动与 Tailscale 互联中心

本 Notebook 专为在 Google Colab GPU 实例中运行设计，具有以下核心特性：
1. **GitHub 仓库同步**：自动从 `https://github.com/bycainiu/zhensuo` 下载与拉取最新代码；
2. **Tailscale 私网专线互联**：仅通过 Tailscale Mesh 私有局域网直连本地电脑（关闭外部公网隧道，安全高速）；
3. **Google Drive 持久化存储**：挂载 `/content/drive/MyDrive/FrameSeek` 自动保存抽取帧、Qdrant 向量索引与工程；
4. **真实 GPU 显存加载与推理加速**：在云端 GPU (T4/A100/L4) 真实加载 Vision-Language 模型，执行多视图特征提取。

## 步骤 1: 检查 Colab GPU 硬件与运行环境

In [1]:
# 1. 检查 GPU 设备与显存分配
!nvidia-smi

import torch
print(f"🔥 PyTorch 版本: {torch.__version__}, CUDA 可用状态: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 当前 GPU 显卡型号: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU 总显存: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

Wed Sep  2 14:34:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 步骤 2: 挂载 Google Drive 进行持久化存储

In [4]:
# 2. 挂载 Google Drive (保存向量数据库、抽帧图像切片与模型缓存)
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
    GDRIVE_BASE = '/content/drive/MyDrive/FrameSeek'
    os.makedirs(f'{GDRIVE_BASE}/models', exist_ok=True)
    os.makedirs(f'{GDRIVE_BASE}/extracted_frames', exist_ok=True)
    os.makedirs(f'{GDRIVE_BASE}/vector_indices', exist_ok=True)
    os.makedirs(f'{GDRIVE_BASE}/export_projects', exist_ok=True)
    print(f"✅ Google Drive 持久化目录初始化成功: {GDRIVE_BASE}")
except Exception as e:
    print(f"⚠️ 挂载提示 (若已挂载可忽略): {e}")

Mounted at /content/drive
✅ Google Drive 持久化目录初始化成功: /content/drive/MyDrive/FrameSeek


## 步骤 3: 从 GitHub (`bycainiu/zhensuo`) 下载并同步项目代码

In [3]:
# 3. 克隆或拉取 GitHub 仓库最新代码 (强制重置同步，避免本地文件冲突)
import os
import sys

REPO_URL = "https://github.com/bycainiu/zhensuo.git"
WORK_DIR = "/content/zhensuo"

if os.path.exists(WORK_DIR) and os.path.exists(os.path.join(WORK_DIR, ".git")):
    print("🔄 检测到已存在项目目录，正在强制同步最新代码 (自动重置覆盖本地缓存)...")
    !cd {WORK_DIR} && git fetch origin main && git reset --hard origin/main && git clean -fd
else:
    print(f"📦 正在克隆 GitHub 仓库: {REPO_URL}...")
    !rm -rf {WORK_DIR}
    !git clone {REPO_URL} {WORK_DIR}

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f"✅ 项目工作区就绪: {WORK_DIR}")

📦 正在克隆 GitHub 仓库: https://github.com/bycainiu/zhensuo.git...
Cloning into '/content/zhensuo'...
remote: Enumerating objects: 671, done.
remote: Counting objects: 100% (671/671), done.
remote: Compressing objects: 100% (580/580), done.
remote: Total 671 (delta 151), reused 581 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (671/671), 12.50 MiB | 9.13 MiB/s, done.
Resolving deltas: 100% (151/151), done.
✅ 项目工作区就绪: /content/zhensuo


## 步骤 4: 安装与启动 Tailscale (与本地电脑组网直连)

In [ ]:
# 4. 安装与配置 Tailscale 私网连接
import os
import subprocess
import time

# 检查是否已安装 tailscale
if subprocess.run(["which", "tailscale"], capture_output=True).returncode != 0:
    print("⬇️ 正在安装 Tailscale...")
    !curl -fsSL https://tailscale.com/install.sh | sh

# 启动 tailscaled 守护进程 (使用 userspace 容器网络模式)
print("🚀 正在后台启动 tailscaled 服务...")
!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 --outbound-http-proxy-listen=localhost:1055 > /content/tailscale.log 2>&1 &
time.sleep(3)

# 可选: 如果有 Tailscale AuthKey 可填在此处实现无感登录，如 "tskey-auth-xxxx"
TAILSCALE_AUTHKEY = ""

if TAILSCALE_AUTHKEY.strip():
    !tailscale up --authkey={TAILSCALE_AUTHKEY} --hostname=colab-zhensuo --accept-routes
else:
    print("\n👉 请点击下方输出中的 Tailscale 授权链接 (或填入 TAILSCALE_AUTHKEY) 登录绑定：\n")
    !tailscale up --hostname=colab-zhensuo --accept-routes

time.sleep(2)
print("\n" + "="*50)
print("🌐 当前 Colab 节点的 Tailscale IP 地址:")
!tailscale ip -4 || echo '暂未分配 IP，请确保已点击上方链接完成 Tailscale 授权'
print("="*50 + "\n")

## 步骤 5: 安装推理与服务依赖

In [ ]:
# 5. 安装 FastAPI、PyTorch、OpenCV 与多模态模型运行依赖
!pip install -q fastapi uvicorn pydantic python-multipart accelerate transformers sentencepiece timm einops requests opencv-python Pillow torchvision
print("✅ 依赖安装完成！")

## 步骤 6: 部署并启动 GPU 模型加速服务 (真实显存加载模式)

In [ ]:
# 6. 使用仓库内 colab/model_server.py (与 GitHub 同步的唯一服务端源码) 启动 GPU 模型服务
import os
import subprocess
import time
import requests

SERVER_SRC = "/content/zhensuo/colab/model_server.py"
if not os.path.isfile(SERVER_SRC):
    raise RuntimeError("未找到 /content/zhensuo/colab/model_server.py，请先运行步骤 3 同步 GitHub 仓库最新代码！")

# 以仓库文件为唯一源复制到 /content 启动 (不再维护 notebook 内嵌副本，避免版本漂移)
with open(SERVER_SRC, "r", encoding="utf-8") as f:
    code = f.read()
with open("/content/model_server.py", "w", encoding="utf-8") as f:
    f.write(code)
print(f"✅ 已加载仓库版服务端代码 ({len(code)} 字符)")

!fuser -k 8000/tcp || true
time.sleep(1)

log_file = open("/content/model_server.log", "w", encoding="utf-8")
proc = subprocess.Popen(["python3", "/content/model_server.py"], stdout=log_file, stderr=subprocess.STDOUT)
print(f"🚀 正在启动后台 GPU 模型服务 (PID: {proc.pid})...")

is_ready = False
for i in range(30):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=1)
        if r.status_code == 200:
            is_ready = True
            break
    except Exception:
        pass

try:
    ts_ip = subprocess.check_output(["tailscale", "ip", "-4"]).decode().strip().split('
')[0]
except Exception:
    ts_ip = "127.0.0.1"

if is_ready:
    print("
" + "="*65)
    print("🎉 FrameSeek GPU 模型加速服务已就绪 (Tailscale 私网模式)！")
    print(f"🔗 【Tailscale 内网直连地址】: http://{ts_ip}:8000")
    print("👉 请在本地前端「模型」页填入该地址并点击探测保存")
    print("="*65)
else:
    print("❌ 服务启动超时，日志如下：")
    with open("/content/model_server.log", "r", encoding="utf-8") as f:
        print(f.read())


## 步骤 7: 测试健康检查 API

In [ ]:
# 7. 发送测试请求确认服务健康度
import requests
import json

try:
    res = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=5)
    print("✅ 服务健康检查响应成功:")
    print(json.dumps(res.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f"⚠️ 健康检查失败: {e}")
    print("\n--- 详细运行日志 (/content/model_server.log) ---")
!cat /content/model_server.log || true

## 步骤 8: 诊断工具与实时显存/内存监控 (新增分析 Cell)

In [ ]:
# 8. 诊断工具：GPU/显存/进程巡检 + 嵌入真实性自测 (核验服务端嵌入的就是给定图片本身)
import requests
import json
import io
import base64
import hashlib
from IPython.display import display, Image as IPyImage

print("=" * 60)
print("📊 FrameSeek 服务端实时诊断")
print("=" * 60)

# 1) 健康检查
try:
    h = requests.get("http://127.0.0.1:8000/api/v1/health", timeout=5).json()
    print(f"✅ 服务健康: device={h.get('device')}, gpu={h.get('gpu')}, "
          f"VRAM {h.get('vram_used_gb')}/{h.get('vram_total_gb')} GB, Drive挂载={h.get('gdrive_connected')}")
except Exception as e:
    print(f"❌ 健康检查失败: {e}")
    print("--- /content/model_server.log 末尾 40 行 ---")
    !tail -40 /content/model_server.log
    raise SystemExit

# 2) 进程与 GPU 实时状态
print("
=== 服务进程 ===")
!ps aux | grep model_server | grep -v grep
print("
=== GPU 实时负荷 ===")
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

# 3) 嵌入真实性自测：构造一张独一无二的测试图，核验服务端识别出的 MD5/字节数与本地图完全一致
from PIL import Image, ImageDraw
img = Image.new("RGB", (640, 360))
d = ImageDraw.Draw(img)
for x in range(640):
    d.line([(x, 0), (x, 360)], fill=(x % 256, (x * 2) % 256, (x * 3) % 256))
d.text((20, 20), "FrameSeek embed self-test 2026", fill=(255, 255, 255))
buf = io.BytesIO()
img.save(buf, format="JPEG", quality=90)
local_bytes = buf.getvalue()
local_md5 = hashlib.md5(local_bytes).hexdigest()

print(f"
🧪 嵌入自测: 本地源图 MD5={local_md5} (字节 {len(local_bytes)})")
try:
    r = requests.post(
        "http://127.0.0.1:8000/api/v1/embed/image",
        json={"image": "data:image/jpeg;base64," + base64.b64encode(local_bytes).decode(),
              "views": ["global", "person_context", "person_tight", "face"]},
        timeout=30,
    )
    j = r.json()
    md5_ok = j.get("image_md5") == local_md5
    bytes_ok = j.get("source_bytes") == len(local_bytes)
    print(f"   服务端返回 MD5={j.get('image_md5')}  {'✅ 一致' if md5_ok else '❌ 不一致！'}")
    print(f"   服务端接收字节={j.get('source_bytes')}  {'✅ 一致' if bytes_ok else '❌ 不一致！'}")
    print(f"   来源解析方式={j.get('source_kind')}, 画幅={j.get('image_dims', {}).get('width')}x{j.get('image_dims', {}).get('height')}")
    print(f"   设备={j.get('gpu_device')}, 延迟={j.get('latency_ms')} ms")

    # 展示服务端神经网络实际输入的 4 视图裁切切片 (肉眼确认嵌入的就是上图)
    print("
🖼️ 服务端实际输入的 4 视图切片:")
    for v, thumb in (j.get("crop_previews") or {}).items():
        print(f"   - {v}:")
        display(IPyImage(base64.b64decode(thumb.split(',', 1)[1])))
except Exception as e:
    print(f"❌ 嵌入自测失败: {e}")


## 步骤 9: 模拟视频多视图 AI 特征抽取推断 (验证 GPU 实时计算)

In [ ]:
# 9. 向本地 FastAPI 发起真实视频多视图抽帧与张量推理测试 (摘要输出 + 真实帧占比核验)
import requests
import time
import json

test_payload = {
    "video_id": "vid_test_colab_gpu",
    "title": "测试 115 视频素材",
    "filename": "test_video.mp4",
    "duration": 120.0,
    "pick_code": "pc_test_123456",  # 假 pickcode：预期 real_frames_count=0, fallback_frames_count>0
}

print("🚀 正在向 GPU 模型服务发送多视图抽帧索引请求...")
t0 = time.time()
try:
    res = requests.post("http://127.0.0.1:8000/api/v1/ingest/process_video", json=test_payload, timeout=60)
    if res.status_code == 200:
        data = res.json()
        print("🎉 GPU 抽取与向量提取完成！")
        print(f"   帧数={data.get('frames_extracted')}, 真实帧={data.get('real_frames_count')}, "
              f"色彩回退帧={data.get('fallback_frames_count')} (回退帧说明 115 未拉到画面，属预期，非真实嵌入)")
        print(f"   设备={data.get('device')}, 延迟={data.get('latency_ms')} ms, 向量数={data.get('vectors_generated')}")
        for fr in data.get("results", [])[:4]:
            print(f"   - {fr.get('frame_id')} @ {fr.get('timestamp')}s real={fr.get('real_frame')} md5={fr.get('frame_md5') or '(回退色块)'}")
    else:
        print(f"❌ 请求返回 HTTP {res.status_code}: {res.text[:500]}")
except Exception as e:
    print(f"❌ 请求异常: {e}")
